In [106]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

Q1. Spark Session & Data Loading

In [107]:
spark = SparkSession.builder \
        .appName("Sales Data Analysis") \
        .getOrCreate()

df = spark.read.csv("/content/sales_data.csv", header=True, inferSchema=True)

df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- order_amount: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)



Q2. Data Type Conversion

In [108]:
df = df.withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
df = df.withColumn("order_timestamp", col("order_timestamp").cast(TimestampType()))
df.show()

+--------+-----------+------+-------+------------+----------+-------------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|
+--------+-----------+------+-------+------------+----------+-------------------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|
+--------+-----------+------+-------+------------+----------+-------------------+



Q3. Overall Sales Metrics

In [17]:
overall_metrics = df.agg(
     sum("order_amount").alias("total_sales"),
     avg("order_amount").alias("avg_orders"),
     max("order_amount").alias("max_sales"),
     min("order_amount").alias("min_sales")
)

overall_metrics.show()

+-----------+----------+---------+---------+
|total_sales|avg_orders|max_sales|min_sales|
+-----------+----------+---------+---------+
|     359000|   44875.0|    80000|    20000|
+-----------+----------+---------+---------+



Q4. Region-wise Analysis

In [21]:
region_sales = df.groupBy("region").agg(
    sum("order_amount").alias("total_sales"),
    round(avg("order_amount"),2).alias("avg_sales"),
    count("order_amount").alias("order_count")
)

region_sales.show()

+------+-----------+---------+-----------+
|region|total_sales|avg_sales|order_count|
+------+-----------+---------+-----------+
| South|     102000|  51000.0|          2|
|  East|     102000|  51000.0|          2|
|  West|      28000|  28000.0|          1|
| North|     127000| 42333.33|          3|
+------+-----------+---------+-----------+



Q5. Customer Count

In [33]:
distinct_customers = df.agg(countDistinct("customer_id").alias("unique_customers"))
distinct_customers.show()

+----------------+
|unique_customers|
+----------------+
|               4|
+----------------+



Q6. Product-wise Aggregation

In [36]:
list_prod = df.groupBy("product").agg(collect_list("order_amount").alias("list"))
set_prod = df.groupBy("product").agg(collect_set("order_amount").alias("set"))
list_prod.show(truncate=False)
set_prod.show(truncate=False)

+-------+---------------------+
|product|list                 |
+-------+---------------------+
|Laptop |[75000, 80000, 72000]|
|Mobile |[30000, 28000, 32000]|
|Tablet |[20000, 22000]       |
+-------+---------------------+

+-------+---------------------+
|product|set                  |
+-------+---------------------+
|Laptop |[72000, 80000, 75000]|
|Mobile |[28000, 30000, 32000]|
|Tablet |[20000, 22000]       |
+-------+---------------------+



Q7. Regional Ranking

In [45]:
from pyspark.sql.window import Window

spec = Window.partitionBy("region").orderBy(col("order_amount").desc())
df = df.withColumn("row_num", row_number().over(spec)) \
       .withColumn("rank", rank().over(spec)) \
       .withColumn("dense_rank", dense_rank().over(spec))

df.select("product","region","rank","dense_rank","row_num").show()

+-------+------+----+----------+-------+
|product|region|rank|dense_rank|row_num|
+-------+------+----+----------+-------+
| Laptop|  East|   1|         1|      1|
| Tablet|  East|   2|         2|      2|
| Laptop| North|   1|         1|      1|
| Mobile| North|   2|         2|      2|
| Tablet| North|   3|         3|      3|
| Laptop| South|   1|         1|      1|
| Mobile| South|   2|         2|      2|
| Mobile|  West|   1|         1|      1|
+-------+------+----+----------+-------+



Q8. Top Orders per Region

In [46]:
from pyspark.sql.window import Window

spec = Window.partitionBy("region").orderBy(col("order_amount").desc())
df_row = df.withColumn("row_num", row_number().over(spec))

df_2= df_row.filter(col("row_num") <= 2)

df_2.select("product","order_amount","region","row_num").show()

+-------+------------+------+-------+
|product|order_amount|region|row_num|
+-------+------------+------+-------+
| Laptop|       80000|  East|      1|
| Tablet|       22000|  East|      2|
| Laptop|       75000| North|      1|
| Mobile|       32000| North|      2|
| Laptop|       72000| South|      1|
| Mobile|       30000| South|      2|
| Mobile|       28000|  West|      1|
+-------+------------+------+-------+



Q9. Order Comparison per Customer

In [49]:

df_with_lag = df.withColumn("prev_order_amount", lag("order_amount", 1).over(spec))
df_with_lag.show()

df_with_lead = df.withColumn("next_order_amount", lead("order_amount", 1).over(spec))
df_with_lead.show()

+--------+-----------+------+-------+------------+----------+-------------------+-------+----+----------+-----------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|row_num|rank|dense_rank|prev_order_amount|
+--------+-----------+------+-------+------------+----------+-------------------+-------+----+----------+-----------------+
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|      1|   1|         1|             NULL|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|      2|   2|         2|            80000|
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|      1|   1|         1|             NULL|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|      2|   2|         2|            75000|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|      3|   3|         3|            32000|
|    100

Q10. Running Total

Q11. Average Order per Customer

In [56]:
cust_spec= Window.partitionBy("customer_id")

df.withColumn("avg_order_per_customer", round(avg("order_amount").over(cust_spec),2)).show()

+--------+-----------+------+-------+------------+----------+-------------------+-------+----+----------+----------------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|row_num|rank|dense_rank|avg_order_per_customer|
+--------+-----------+------+-------+------------+----------+-------------------+-------+----+----------+----------------------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|      1|   1|         1|              42333.33|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|      2|   2|         2|              42333.33|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|      3|   3|         3|              42333.33|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|      1|   1|         1|               51000.0|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|      2|   2|   

Q12. Maximum Order per Region

In [57]:
reg_spec= Window.partitionBy("region")

df.withColumn("avg_order_per_customer", round(avg("order_amount").over(reg_spec),2)).show()

+--------+-----------+------+-------+------------+----------+-------------------+-------+----+----------+----------------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|row_num|rank|dense_rank|avg_order_per_customer|
+--------+-----------+------+-------+------------+----------+-------------------+-------+----+----------+----------------------+
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|      1|   1|         1|               51000.0|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|      2|   2|         2|               51000.0|
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|      1|   1|         1|              42333.33|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|      2|   2|         2|              42333.33|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|      3|   3|   

Q13. Date Components Extraction

In [62]:
df_dates = df.withColumn("year", year(col("order_date"))) \
             .withColumn("month", month(col("order_date"))) \
             .withColumn("day", dayofmonth(col("order_date")))

df_dates.select("order_date","year","month","day").show()

+----------+----+-----+---+
|order_date|year|month|day|
+----------+----+-----+---+
|2023-01-10|2023|    1| 10|
|2023-01-12|2023|    1| 12|
|2023-02-05|2023|    2|  5|
|2023-02-20|2023|    2| 20|
|2023-03-01|2023|    3|  1|
|2023-03-15|2023|    3| 15|
|2023-03-18|2023|    3| 18|
|2023-04-02|2023|    4|  2|
+----------+----+-----+---+



Q14. Date Difference

In [68]:
df_dates = df.withColumn("days_since_order",datediff(current_date(),"order_date"))

df_dates.select("order_date","days_since_order").show()

+----------+----------------+
|order_date|days_since_order|
+----------+----------------+
|2023-01-10|            1171|
|2023-01-12|            1169|
|2023-02-05|            1145|
|2023-02-20|            1130|
|2023-03-01|            1121|
|2023-03-15|            1107|
|2023-03-18|            1104|
|2023-04-02|            1089|
+----------+----------------+



Q15. Create new columns

In [83]:
df_dates = df_dates.withColumn("order_year", year("order_date")) \
                   .withColumn("order_month", month("order_date")) \
                   .withColumn("week", weekofyear(col("order_date")))

df_dates.select("order_date", "order_year", "order_month", "week").show()


+----------+----------+-----------+----+
|order_date|order_year|order_month|week|
+----------+----------+-----------+----+
|2023-01-10|      2023|          1|   2|
|2023-01-12|      2023|          1|   2|
|2023-02-05|      2023|          2|   5|
|2023-02-20|      2023|          2|   8|
|2023-03-01|      2023|          3|   9|
|2023-03-15|      2023|          3|  11|
|2023-03-18|      2023|          3|  11|
|2023-04-02|      2023|          4|  13|
+----------+----------+-----------+----+



Q16. Date-based Filtering

In [84]:
df_selected_date = df.filter((col("order_date") >= "2023-01-01") & (col("order_date") <= "2023-03-31"))
df_selected_date.show()

+--------+-----------+------+-------+------------+----------+-------------------+-------+----+----------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|row_num|rank|dense_rank|
+--------+-----------+------+-------+------------+----------+-------------------+-------+----+----------+
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|      1|   1|         1|
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|      1|   1|         1|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|      2|   2|         2|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|      3|   3|         3|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|      1|   1|         1|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|      2|   2|         2|
|    1005|       C004|  West| Mobile|       28

Q17. Monthly Sales Trend

In [86]:
df = df.withColumn("year", year(col("order_date"))) \
       .withColumn("month", month(col("order_date")))

monthly_sales = df.groupBy("year", "month") \
                  .agg(sum("order_amount").alias("total_sales")) \
                  .orderBy("year", "month")

monthly_sales.show()

+----+-----+-----------+
|year|month|total_sales|
+----+-----+-----------+
|2023|    1|     105000|
|2023|    2|     100000|
|2023|    3|     132000|
|2023|    4|      22000|
+----+-----+-----------+



Q18. Customer Activity Analysis

In [109]:

cust_order_count = df.groupBy("customer_id") \
                            .agg(count("order_id").alias("total_orders"))

customers_more_than_2_orders = cust_order_count.filter(col("total_orders") > 2)

customers_more_than_2_orders.show()

+-----------+------------+
|customer_id|total_orders|
+-----------+------------+
|       C001|           3|
+-----------+------------+

